# Aumento de los datos (Data Augmentation) en PyTorch

Las redes neuronales profundas necesitan muchos datos para aprender bien. Cuando tenemos pocos, aparece el **sobreajuste**: la red memoriza las imágenes de entrenamiento en lugar de aprender a reconocer gatos y perros en general. Ya lo vimos en el notebook de *transfer learning*: aun con un modelo pre-entrenado y pocos parámetros entrenables, la pérdida de validación se estanca mientras la de entrenamiento sigue bajando.

Hemos discutido diferentes estrategias contra el sobreajuste, como el dropout y la regularización. En este notebook estudiamos otra: el **aumento de datos** (*Data Augmentation*), que consiste en generar imágenes sintéticas a partir de alteraciones de las imágenes del conjunto de entrenamiento. Un gato volteado horizontalmente o rotado unos grados sigue siendo un gato, pero para la red es una imagen distinta.

Usaremos el mismo problema (clasificar **gatos** y **perros**, base de datos [Cats vs Dogs](https://www.tensorflow.org/datasets/catalog/cats_vs_dogs?hl=es-419)) y el mismo modelo del notebook anterior (VGG16 congelada con capas nuevas), para poder **comparar** con y sin aumento de datos.

**Al terminar este notebook podrás:**

- Definir transformaciones de aumento de datos con `torchvision.transforms`.
- Aplicarlas solo al conjunto de entrenamiento.
- Entrenar con y sin aumento de datos y comparar el efecto sobre el sobreajuste.

En este notebook usaremos **PyTorch**. A diferencia de otras librerías que ocultan el ciclo de entrenamiento dentro de una función (`fit`), en PyTorch lo escribimos nosotros mismos, lo que nos permite ver exactamente qué ocurre en cada paso del aprendizaje.

## Etapa 1: Carga de los datos

Se cargan las librerías necesarias y se elige el dispositivo (GPU si está disponible). Fijamos también una semilla (`manual_seed`) para que los resultados sean reproducibles.

In [ ]:
import os
import urllib.request
import zipfile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0)
print('Dispositivo:', device)

Cats vs Dogs **no** viene incluida en `torchvision`. Si ya ejecutaste el notebook anterior, los datos ya están descargados y este paso se salta. Si no, se descargan (unos 790 MB) y se descomprimen en una carpeta con una subcarpeta por clase (`Cat` y `Dog`), que leemos con `datasets.ImageFolder`.

In [ ]:
data_dir = '../../../data'
url = 'https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip'
zip_path = os.path.join(data_dir, 'cats_and_dogs.zip')
images_dir = os.path.join(data_dir, 'PetImages')

os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(images_dir):
    print('Descargando...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(data_dir)
    os.remove(zip_path)
print(os.listdir(images_dir))

La base de datos contiene algunas imágenes dañadas. Para que el entrenamiento no se detenga con un error, le pasamos a `ImageFolder` una función que descarta las que no se pueden abrir (tarda un par de minutos).

In [ ]:
def is_valid_image(path):
    if not path.lower().endswith('.jpg'):
        return False
    try:
        with Image.open(path) as img:
            img.load()  # fuerza la lectura completa de la imagen
        return True
    except Exception:
        return False

# transform=None: por ahora cada elemento es una imagen PIL; las transformaciones se agregan después.
full_data = datasets.ImageFolder(images_dir, transform=None, is_valid_file=is_valid_image)
class_names = full_data.classes
print('Clases:', class_names)
print('Imágenes válidas:', len(full_data))

Separamos tres conjuntos, igual que en el notebook de clasificación de dígitos:

| Conjunto | Proporción | Para qué sirve |
|---|---|---|
| Entrenamiento | 70 % | Calcular los pesos de la red |
| Validación | 15 % | Vigilar el desempeño época a época |
| Prueba (test) | 15 % | Evaluación final, una sola vez, al terminar |

Como la base de datos viene en un solo bloque, la dividimos nosotros con `random_split`. Fijamos una semilla para que la división sea siempre la misma.

In [ ]:
generator = torch.Generator().manual_seed(42)
n_total = len(full_data)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val
train_subset, val_subset, test_subset = random_split(full_data, [n_train, n_val, n_test], generator=generator)
print('Entrenamiento:', len(train_subset), '| Validación:', len(val_subset), '| Prueba:', len(test_subset))

## Etapa 2: Aumento de los datos

En PyTorch, el aumento de datos se define como una serie de transformaciones **aleatorias** que se aplican a cada imagen cada vez que el `DataLoader` la lee. Así, en cada época la red ve una versión ligeramente diferente de la misma imagen, sin que el conjunto de datos en disco cambie. Usamos:

- `RandomHorizontalFlip`: voltea la imagen horizontalmente con probabilidad 0.5.
- `RandomRotation(36)`: rota la imagen un ángulo aleatorio entre -36° y 36°.

Estas transformaciones operan sobre la imagen original (antes de convertirla en tensor). También redimensionamos a $150 \times 150$ píxeles ($H = W = 150$), porque la red necesita entradas del mismo tamaño.

In [ ]:
augment = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(36),
])

Veamos el efecto: aplicamos la misma transformación 9 veces a una misma imagen. Cada llamada produce una versión distinta.

In [ ]:
original_image, label = train_subset[0]

plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(augment(original_image))
    plt.title(class_names[label])
    plt.axis('off')
plt.tight_layout()
plt.show()

Ahora definimos las transformaciones completas de cada conjunto. Además del aumento, hacemos dos conversiones:

1. `ToTensor()` convierte la imagen en un tensor y normaliza los píxeles del rango $[0, 255]$ a $[0, 1]$. También reordena las dimensiones a `(canales, H, W)`.
2. `Normalize(mean, std)` usa las estadísticas de ImageNet. **Es obligatorio al usar un modelo pre-entrenado:** VGG16 fue entrenada con imágenes normalizadas así.

**Importante:** el aumento de datos se aplica **solo al conjunto de entrenamiento**. Los conjuntos de validación y prueba usan únicamente `preprocess` (redimensionar y normalizar), pues queremos medir el rendimiento sobre imágenes reales y sin alteraciones.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Sin aumento: solo redimensionar y normalizar
preprocess = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# Con aumento: primero las alteraciones aleatorias, luego las mismas conversiones
train_transform_aug = transforms.Compose([
    augment,
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

`random_split` nos entrega subconjuntos que comparten una misma base de datos, y esa base no tiene transformaciones. Para poder aplicar una transformación distinta al conjunto de entrenamiento y otra distinta a los demás, definimos una pequeña clase `TransformedSubset`. Una clase `Dataset` de PyTorch solo necesita dos métodos: `__len__` (cuántos datos hay) y `__getitem__` (cómo obtener el dato número `i`).

In [ ]:
class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        image, label = self.subset[i]
        return self.transform(image), label

Creamos los `DataLoader`, que entregan los datos en **minibatches** de tamaño 32. Tendremos dos para entrenamiento: uno sin aumento y otro con aumento. Los de validación y prueba son únicos.

In [ ]:
batch_size = 32

def make_loader(subset, transform, shuffle):
    return DataLoader(TransformedSubset(subset, transform), batch_size=batch_size,
                      shuffle=shuffle, num_workers=2)

train_loader = make_loader(train_subset, preprocess, shuffle=True)            # sin aumento
train_loader_aug = make_loader(train_subset, train_transform_aug, shuffle=True)  # con aumento
val_loader = make_loader(val_subset, preprocess, shuffle=False)
test_loader = make_loader(test_subset, preprocess, shuffle=False)

images, labels = next(iter(train_loader_aug))
print(images.shape)  # (batch, canales, H, W)
print(labels.shape)

## Etapa 3: El modelo

Usamos el mismo modelo del notebook de *transfer learning*: la parte convolucional de VGG16 pre-entrenada en ImageNet y **congelada** (`requires_grad = False`), seguida de capas nuevas (aplastar, densa de 256 unidades, Dropout y una salida de un solo logit). Como ya lo conocemos, lo definimos directamente como una **clase** con un método por cada tarea:

| Método | Qué hace |
|---|---|
| `__init__` | Carga el modelo base, lo congela, agrega las capas nuevas y crea la pérdida y el optimizador. |
| `train_epoch` | Recorre los minibatches de una época y actualiza los pesos. |
| `evaluate` | Mide pérdida y accuracy sobre un conjunto, sin entrenar. |
| `fit` | Repite `train_epoch` y `evaluate` por varias épocas y guarda el historial. |
| `predict` | Devuelve la clase predicha para unas imágenes. |

Los pasos de cada iteración son: `zero_grad`, forward, pérdida, `backward` y `step`. El accuracy se calcula comparando el signo del logit (mayor que 0 = "perro") con la etiqueta.

In [ ]:
def accuracy(logits, y):
    # Un logit mayor que 0 equivale a una probabilidad mayor que 0.5: se predice "perro" (clase 1)
    return ((logits > 0).float() == y).float().mean().item()

In [ ]:
class CatsDogsClassifier:
    def __init__(self, hidden_units=256, dropout=0.5, lr=1e-3):
        # Modelo base: la parte convolucional de VGG16 pre-entrenada en ImageNet, congelada
        base_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        for param in base_model.parameters():
            param.requires_grad = False

        # Capas nuevas: son las únicas que se entrenan
        self.net = nn.Sequential(
            base_model,
            nn.Flatten(),                         # 512 x 4 x 4 = 8192 unidades
            nn.Linear(512 * 4 * 4, hidden_units),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_units, 1),           # un solo logit: qué tan probable es "perro"
        ).to(device)

        self.loss_fn = nn.BCEWithLogitsLoss()
        # Al optimizador solo le pasamos los parámetros que no están congelados
        trainable = [p for p in self.net.parameters() if p.requires_grad]
        self.optimizer = torch.optim.Adam(trainable, lr=lr)
        self.history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

    def train_epoch(self, loader):
        # Una época: recorre todos los minibatches y actualiza los pesos
        self.net.train()
        total_loss, total_acc, n = 0.0, 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device).float()
            self.optimizer.zero_grad()
            logits = self.net(xb).squeeze(1)
            loss = self.loss_fn(logits, yb)
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * len(yb)
            total_acc += accuracy(logits, yb) * len(yb)
            n += len(yb)
        return total_loss / n, total_acc / n

    def evaluate(self, loader):
        # Mide el desempeño sin entrenar (sin calcular gradientes)
        self.net.eval()
        total_loss, total_acc, n = 0.0, 0.0, 0
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device).float()
                logits = self.net(xb).squeeze(1)
                total_loss += self.loss_fn(logits, yb).item() * len(yb)
                total_acc += accuracy(logits, yb) * len(yb)
                n += len(yb)
        return total_loss / n, total_acc / n

    def fit(self, train_loader, val_loader, epochs=20):
        pbar = tqdm(range(epochs), desc='Entrenamiento', unit='época')
        for epoch in pbar:
            loss, acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.evaluate(val_loader)
            for key, value in zip(self.history, (loss, acc, val_loss, val_acc)):
                self.history[key].append(value)
            pbar.set_postfix(loss=f'{loss:.4f}', accuracy=f'{acc:.4f}',
                             val_loss=f'{val_loss:.4f}', val_accuracy=f'{val_acc:.4f}')

    def predict(self, x):
        # Devuelve la clase predicha (0 = gato, 1 = perro) para cada imagen del minibatch
        self.net.eval()
        with torch.no_grad():
            return (self.net(x.to(device)).squeeze(1) > 0).long().cpu()

## Etapa 4: Entrenamiento con y sin aumento de datos

Para que la comparación sea justa, creamos **dos modelos nuevos** (cada instancia de la clase parte del mismo modelo base y de capas nuevas con pesos aleatorios) y los entrenamos igual, salvo por el `DataLoader` de entrenamiento. Entrenamos 20 épocas con minibatches de 32. En GPU cada modelo tarda unos minutos; en CPU puede tardar bastante más, así que puedes reducir `epochs` para probar.

Primero, el modelo **sin** aumento de datos:

In [ ]:
epochs = 20

clf = CatsDogsClassifier()
clf.fit(train_loader, val_loader, epochs=epochs)

Ahora el modelo **con** aumento de datos. Cada vez que el `DataLoader` lee una imagen de entrenamiento, la voltea y rota al azar:

In [ ]:
clf_aug = CatsDogsClassifier()
clf_aug.fit(train_loader_aug, val_loader, epochs=epochs)

## Etapa 5: Comparación

Graficamos las curvas de ambos modelos. Qué buscar:

- La **brecha** entre la pérdida de entrenamiento y la de validación. Una brecha grande indica sobreajuste.
- Si la pérdida de validación del modelo sin aumento empieza a subir mientras la de entrenamiento baja.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, model, title in [(axes[0], clf, 'Sin aumento de datos'), (axes[1], clf_aug, 'Con aumento de datos')]:
    ax.plot(model.history['loss'], label='entrenamiento')
    ax.plot(model.history['val_loss'], label='validación')
    ax.set_title(title)
    ax.set_xlabel('época')
    ax.set_ylabel('pérdida')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(clf.history['val_accuracy'], label='sin aumento')
plt.plot(clf_aug.history['val_accuracy'], label='con aumento')
plt.title('Accuracy en validación')
plt.xlabel('época')
plt.ylabel('accuracy')
plt.legend()
plt.tight_layout()
plt.show()

Con el aumento de datos, la brecha entre entrenamiento y validación suele ser menor: el modelo generaliza mejor porque nunca ve exactamente la misma imagen dos veces. Observa también que la pérdida y el accuracy de **entrenamiento** son algo peores con aumento. Eso es esperado: la tarea es más difícil, y esa dificultad es justamente lo que evita que la red memorice.

Finalmente, evaluamos ambos modelos en el conjunto de **prueba**, que no se usó ni para entrenar ni para vigilar el entrenamiento. Es la comparación definitiva.

In [ ]:
_, acc = clf.evaluate(test_loader)
_, acc_aug = clf_aug.evaluate(test_loader)
print(f'Accuracy en prueba sin aumento de datos: {acc:.4f}')
print(f'Accuracy en prueba con aumento de datos: {acc_aug:.4f}')

## Resumen

- El **aumento de datos** genera variaciones de las imágenes de entrenamiento (volteos, rotaciones, etc.) y es una forma de reducir el sobreajuste.
- En PyTorch se define con transformaciones **aleatorias** de `torchvision.transforms`, que se aplican cada vez que se lee una imagen: en cada época la red ve versiones distintas.
- Se aplica **solo al conjunto de entrenamiento**. Validación y prueba se dejan sin alterar.
- Para comparar de forma justa, cada experimento parte de un modelo nuevo.
- El aumento de datos y la transferencia de aprendizaje se complementan: permiten aprovechar redes profundas cuando los datos etiquetados y los recursos computacionales son escasos.

## Ejercicios

Modifica un valor a la vez, vuelve a entrenar un modelo con aumento y compáralo con `clf`.

1. Cambia `RandomRotation(36)` a `RandomRotation(10)` y luego a `RandomRotation(90)`. ¿Qué tan fuerte puede ser la rotación antes de que empiece a perjudicar?
2. Agrega `transforms.RandomVerticalFlip()`. ¿Tiene sentido para fotos de gatos y perros? ¿Y para otro tipo de imágenes, como imágenes satelitales?
3. Agrega `transforms.ColorJitter(brightness=0.3, contrast=0.3)` dentro de `augment`. ¿Cambia el resultado?
4. Agrega `transforms.RandomResizedCrop(150, scale=(0.7, 1.0))` en lugar de `Resize`. ¿Qué hace esta transformación?
5. **Para pensar:** ¿por qué NO aplicamos el aumento de datos al conjunto de prueba?